# Análisis de Votaciones — Cámara de Representantes, Valle del Cauca

**Candidata de interés:** Ana María Sanclemente (Candidato #107, Partido CAMBIO RADICAL/ALMA #3050)

**Objetivo:** Cruzar los resultados electorales de Cámara con la base de simpatizantes para medir la efectividad de la estrategia territorial por puesto de votación, zona/comuna, municipio y líder.

---

In [ ]:
# === SETUP Y DEPENDENCIAS ===
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

# --- Constantes ---
CANDIDATA_CODE = 107
CANDIDATA_NOMBRE = 'Ana Maria Sanclemente'
PARTIDO_CODE = 3050
PARTIDO_NOMBRE = 'CAMBIO RADICAL/ALMA'

# Mapeo de codigos electorales -> nombres de municipio
MPIO_MAP = {
    1: 'CALI', 4: 'ALCALA', 7: 'ANDALUCIA', 10: 'ANSERMANUEVO',
    13: 'ARGELIA', 16: 'BOLIVAR', 19: 'BUENAVENTURA', 22: 'BUGA',
    25: 'BUGALAGRANDE', 28: 'CAICEDONIA', 31: 'CALIMA', 34: 'CANDELARIA',
    37: 'CARTAGO', 40: 'DAGUA', 43: 'EL AGUILA', 46: 'EL CAIRO',
    49: 'EL CERRITO', 52: 'EL DOVIO', 55: 'FLORIDA', 58: 'GINEBRA',
    61: 'GUACARI', 64: 'JAMUNDI', 67: 'LA CUMBRE', 70: 'LA UNION',
    73: 'LA VICTORIA', 76: 'OBANDO', 79: 'PALMIRA', 82: 'PRADERA',
    85: 'RESTREPO', 88: 'RIOFRIO', 91: 'ROLDANILLO', 94: 'SAN PEDRO',
    97: 'SEVILLA', 100: 'TORO', 103: 'TRUJILLO', 106: 'TULUA',
    109: 'ULLOA', 112: 'VERSALLES', 115: 'VIJES', 118: 'YOTOCO',
    121: 'YUMBO', 124: 'ZARZAL'
}
MPIO_NAME_TO_CODE = {v: k for k, v in MPIO_MAP.items()}

# Mapeo de codigos de partidos -> nombres
PARTIDO_MAP = {
    1: 'LIBERAL', 2: 'CONSERVADOR', 4: 'ALIANZA VERDE', 5: 'AICO',
    8: 'LA U', 11: 'CENTRO DEMOCRATICO',
    20: 'SALVACION NACIONAL', 21: 'PARTIDO OXIGENO',
    24: 'DEMOCRATA COLOMBIANO',
    3050: 'CAMBIO R-ALMA', 3051: 'MIRA-NVO LIB',
    3057: 'PACTO HISTORICO', 3113: 'FRENTE AMPLIO', 3137: 'FUERZA CIUDADANA'
}

print('Setup completado')

## 1. Carga y Validacion de Datos

In [ ]:
# === CARGAR DATOS ===
df_camara = pd.read_excel('CAMARA valle.xlsx', sheet_name='CAMARA_valle')
df_simpatizantes = pd.read_excel('Registros_simpatizantes_13Mar.xlsx')

print('=== CAMARA valle.xlsx ===')
print(f'Registros: {len(df_camara):,}')
print(f'Columnas: {list(df_camara.columns)}')
print(f'Municipios: {df_camara["mpio"].nunique()}')
print(f'Total votos registrados: {df_camara["votos"].sum():,}')
display(df_camara.head())

print('\n=== Registros_simpatizantes_13Mar.xlsx ===')
print(f'Registros: {len(df_simpatizantes):,}')
print(f'Columnas: {list(df_simpatizantes.columns)}')
print(f'Sin departamento: {df_simpatizantes["Departamento"].isna().sum()}')
display(df_simpatizantes.head())

## 2. Limpieza y Preparacion de Datos

In [ ]:
# === LIMPIEZA Y ENRIQUECIMIENTO ===

# --- CAMARA: agregar nombres legibles ---
df_camara['municipio_nombre'] = df_camara['mpio'].map(MPIO_MAP)
df_camara['partido_nombre'] = df_camara['partido'].map(PARTIDO_MAP).fillna('OTRO')

# Excluir votos especiales (en blanco=996, nulos=997, no marcados=998)
# y candidato 0 (votos por lista, no preferente)
CODIGOS_ESPECIALES = {996: 'VOTO EN BLANCO', 997: 'VOTOS NULOS', 998: 'TARJETAS NO MARCADAS', 0: 'VOTO POR LISTA'}
df_camara['tipo_voto'] = df_camara['candidato'].map(CODIGOS_ESPECIALES).fillna('PREFERENTE')

# --- SIMPATIZANTES: limpiar y normalizar ---
df_sim = df_simpatizantes.copy()
df_sim['Municipio'] = df_sim['Municipio'].astype(str).str.strip().str.upper()
df_sim['Lugar'] = df_sim['Lugar'].astype(str).str.strip().str.upper()
df_sim['Departamento'] = df_sim['Departamento'].astype(str).str.strip().str.upper()
df_sim.loc[df_sim['Municipio'] == 'NAN', 'Municipio'] = None
df_sim.loc[df_sim['Lugar'] == 'NAN', 'Lugar'] = None
df_sim.loc[df_sim['Departamento'] == 'NAN', 'Departamento'] = None

# Normalizar nombres de municipio para que coincidan con MPIO_MAP
mpio_fixes = {'CALIMA (DARIEN)': 'CALIMA', 'CALIMA(DARIEN)': 'CALIMA'}
df_sim['Municipio'] = df_sim['Municipio'].replace(mpio_fixes)

# Agregar codigo de municipio para el cruce
df_sim['mpio_code'] = df_sim['Municipio'].map(MPIO_NAME_TO_CODE)
df_sim['zona_code'] = pd.to_numeric(df_sim['Comuna'], errors='coerce')

# Filtrar simpatizantes del Valle del Cauca (donde hay datos de CAMARA)
df_sim_valle = df_sim[df_sim['Departamento'] == 'VALLE'].copy()
df_sim_fuera = df_sim[df_sim['Departamento'] != 'VALLE']

print(f'Simpatizantes en Valle del Cauca: {len(df_sim_valle):,}')
print(f'Simpatizantes fuera del Valle: {len(df_sim_fuera):,} (no se cruzan con CAMARA)')
print(f'Sin departamento: {df_sim["Departamento"].isna().sum()}')
print(f'\nMunicipios con simpatizantes: {df_sim_valle["Municipio"].nunique()}')
print(f'Puestos (Lugares) unicos: {df_sim_valle["Lugar"].nunique()}')
print(f'Lideres unicos: {df_sim_valle["Líder"].nunique()}')

## 3. Tablas Agregadas

El archivo CAMARA usa codigos numericos para los puestos de votacion (`pto`), mientras que Simpatizantes tiene los nombres textuales (`Lugar`). El cruce se realiza a nivel de:
- **Municipio**: `mpio` (codigo) <-> `Municipio` (nombre)
- **Zona/Comuna**: `zona` (codigo) <-> `Comuna` (numero)

Esto permite un analisis robusto por zona y municipio.

In [ ]:
# === TABLA 1: VOTOS POR ZONA (Municipio + Zona) ===

# Solo votos preferentes de candidatos reales (excluir especiales)
df_preferente = df_camara[df_camara['tipo_voto'] == 'PREFERENTE']

# Votos totales por zona (todos los candidatos)
votos_zona_total = (df_camara
    .groupby(['mpio', 'municipio_nombre', 'zona'])['votos']
    .sum()
    .reset_index()
    .rename(columns={'votos': 'votos_totales_zona'})
)

# Votos de Ana Maria Sanclemente por zona (SOLO PARTIDO 3050)
votos_candidata_zona = (df_camara[(df_camara['candidato'] == CANDIDATA_CODE) & (df_camara['partido'] == PARTIDO_CODE)]
    .groupby(['mpio', 'municipio_nombre', 'zona'])['votos']
    .sum()
    .reset_index()
    .rename(columns={'votos': 'votos_candidata_zona'})
)

# Votos del partido CAMBIO R-ALMA por zona (todos sus candidatos)
votos_partido_zona = (df_camara[df_camara['partido'] == PARTIDO_CODE]
    .groupby(['mpio', 'municipio_nombre', 'zona'])['votos']
    .sum()
    .reset_index()
    .rename(columns={'votos': 'votos_partido_zona'})
)

# === TABLA 2: SIMPATIZANTES POR ZONA ===
sim_zona = (df_sim_valle
    .groupby(['Municipio', 'zona_code'])
    .agg(
        simpatizantes=('Cédula', 'count'),
        puestos_unicos=('Lugar', 'nunique'),
        lideres_unicos=('Líder', 'nunique')
    )
    .reset_index()
)

# === MERGE PRINCIPAL: Zona level ===
# Primero merge votos
votos_merged = votos_zona_total.merge(
    votos_candidata_zona, on=['mpio', 'municipio_nombre', 'zona'], how='left'
).merge(
    votos_partido_zona, on=['mpio', 'municipio_nombre', 'zona'], how='left'
)
votos_merged['votos_candidata_zona'] = votos_merged['votos_candidata_zona'].fillna(0).astype(int)
votos_merged['votos_partido_zona'] = votos_merged['votos_partido_zona'].fillna(0).astype(int)

# Merge con simpatizantes
analisis_zona = votos_merged.merge(
    sim_zona,
    left_on=['municipio_nombre', 'zona'],
    right_on=['Municipio', 'zona_code'],
    how='outer',
    indicator=True
)

# Limpiar
analisis_zona['simpatizantes'] = analisis_zona['simpatizantes'].fillna(0).astype(int)
analisis_zona['votos_totales_zona'] = analisis_zona['votos_totales_zona'].fillna(0).astype(int)
analisis_zona['votos_candidata_zona'] = analisis_zona['votos_candidata_zona'].fillna(0).astype(int)
analisis_zona['municipio_nombre'] = analisis_zona['municipio_nombre'].fillna(analisis_zona['Municipio'])

# Calcular metricas
analisis_zona['ratio_candidata_vs_simpatizantes'] = (
    analisis_zona['votos_candidata_zona'] / analisis_zona['simpatizantes'].replace(0, float('nan'))
).round(2)
analisis_zona['pct_candidata_en_zona'] = (
    analisis_zona['votos_candidata_zona'] / analisis_zona['votos_totales_zona'].replace(0, float('nan')) * 100
).round(2)

# Mostrar zonas con simpatizantes
zona_con_simp = analisis_zona[analisis_zona['simpatizantes'] > 0].sort_values('simpatizantes', ascending=False)
display_cols = ['municipio_nombre', 'zona', 'simpatizantes', 'votos_totales_zona',
                'votos_candidata_zona', 'votos_partido_zona',
                'ratio_candidata_vs_simpatizantes', 'pct_candidata_en_zona']
zona_con_simp[[c for c in display_cols if c in zona_con_simp.columns]].head(20)

## 4. Consolidado por Municipio

In [ ]:
# =================================================================
# RESUMEN POR MUNICIPIO
# =================================================================

# Votos totales por municipio
votos_mpio_total = df_camara.groupby(['mpio', 'municipio_nombre'])['votos'].sum().reset_index()
votos_mpio_total.columns = ['mpio', 'municipio', 'votos_totales']

# Votos candidata por municipio (SOLO PARTIDO 3050)
votos_mpio_candidata = (df_camara[(df_camara['candidato'] == CANDIDATA_CODE) & (df_camara['partido'] == PARTIDO_CODE)]
    .groupby(['mpio', 'municipio_nombre'])['votos'].sum().reset_index())
votos_mpio_candidata.columns = ['mpio', 'municipio', 'votos_candidata']

# Votos partido por municipio
votos_mpio_partido = (df_camara[df_camara['partido'] == PARTIDO_CODE]
    .groupby(['mpio', 'municipio_nombre'])['votos'].sum().reset_index())
votos_mpio_partido.columns = ['mpio', 'municipio', 'votos_partido']

# Simpatizantes por municipio
sim_mpio = (df_sim_valle.groupby('Municipio')
    .agg(
        simpatizantes=('Cédula', 'count'),
        puestos_unicos=('Lugar', 'nunique'),
        lideres_unicos=('Líder', 'nunique'),
        zonas_unicos=('zona_code', 'nunique')
    ).reset_index()
    .rename(columns={'Municipio': 'municipio'})
)

# Merge
resumen_mpio = votos_mpio_total.merge(votos_mpio_candidata[['municipio', 'votos_candidata']], on='municipio', how='left')
resumen_mpio = resumen_mpio.merge(votos_mpio_partido[['municipio', 'votos_partido']], on='municipio', how='left')
resumen_mpio = resumen_mpio.merge(sim_mpio, on='municipio', how='outer')
resumen_mpio['votos_candidata'] = resumen_mpio['votos_candidata'].fillna(0).astype(int)
resumen_mpio['votos_partido'] = resumen_mpio['votos_partido'].fillna(0).astype(int)
resumen_mpio['simpatizantes'] = resumen_mpio['simpatizantes'].fillna(0).astype(int)

# Metricas
resumen_mpio['ratio_votos_vs_simp'] = (
    resumen_mpio['votos_candidata'] / resumen_mpio['simpatizantes'].replace(0, float('nan'))
).round(2)
resumen_mpio['pct_candidata'] = (
    resumen_mpio['votos_candidata'] / resumen_mpio['votos_totales'].replace(0, float('nan')) * 100
).round(2)

# Ordenar por simpatizantes
resumen_mpio = resumen_mpio.sort_values('simpatizantes', ascending=False)

# Totales
total_row = pd.DataFrame([{
    'municipio': '** TOTAL **',
    'votos_totales': resumen_mpio['votos_totales'].sum(),
    'votos_candidata': resumen_mpio['votos_candidata'].sum(),
    'votos_partido': resumen_mpio['votos_partido'].sum(),
    'simpatizantes': resumen_mpio['simpatizantes'].sum(),
    'ratio_votos_vs_simp': round(resumen_mpio['votos_candidata'].sum() / max(resumen_mpio['simpatizantes'].sum(), 1), 2),
    'pct_candidata': round(resumen_mpio['votos_candidata'].sum() / max(resumen_mpio['votos_totales'].sum(), 1) * 100, 2),
}])
resumen_display = pd.concat([resumen_mpio, total_row], ignore_index=True)

cols = ['municipio', 'simpatizantes', 'votos_totales', 'votos_candidata', 'votos_partido', 'ratio_votos_vs_simp', 'pct_candidata']
print(f'=== RESUMEN POR MUNICIPIO - {CANDIDATA_NOMBRE} (Solo Partido 3050) ===\n')
display(resumen_display[cols])

## 5. Analisis por Puesto de Votacion (Simpatizantes)

In [ ]:
# === SIMPATIZANTES POR PUESTO DE VOTACION ===
# Cada puesto (Lugar) tiene un nombre texto en la base de simpatizantes

sim_por_puesto = (df_sim_valle
    .groupby(['Municipio', 'Lugar', 'zona_code'])
    .agg(
        simpatizantes=('Cédula', 'count'),
        lideres=('Líder', 'nunique'),
        lista_lideres=('Líder', lambda x: ', '.join(sorted(x.unique())))
    )
    .reset_index()
    .sort_values('simpatizantes', ascending=False)
)

# Agregar votos de la zona correspondiente para contexto
sim_por_puesto = sim_por_puesto.merge(
    analisis_zona[['municipio_nombre', 'zona', 'votos_totales_zona', 'votos_candidata_zona', 'votos_partido_zona']],
    left_on=['Municipio', 'zona_code'],
    right_on=['municipio_nombre', 'zona'],
    how='left'
)

print(f'=== TOP 40 PUESTOS CON MAS SIMPATIZANTES ===\n')
cols_puesto = ['Municipio', 'Lugar', 'zona_code', 'simpatizantes', 'votos_totales_zona', 'votos_candidata_zona', 'lista_lideres']
display(sim_por_puesto[cols_puesto].head(40))

## 6. Efectividad por Lider

In [ ]:
# === EFECTIVIDAD POR LIDER ===
# Cada simpatizante tiene asignado un lider. Analizamos cuantos simpatizantes
# maneja cada lider, en cuantas zonas opera, y cuantos votos obtuvo la candidata
# en esas zonas.

# Simpatizantes por lider
lider_sim = (df_sim_valle
    .groupby('Líder')
    .agg(
        simpatizantes=('Cédula', 'count'),
        municipios=('Municipio', 'nunique'),
        puestos=('Lugar', 'nunique'),
        zonas=('zona_code', 'nunique'),
        lista_municipios=('Municipio', lambda x: ', '.join(sorted(x.unique())))
    )
    .reset_index()
    .sort_values('simpatizantes', ascending=False)
)

# Para cada lider, sumar los votos de la candidata en las zonas donde tiene simpatizantes
lider_zonas = (df_sim_valle
    .groupby(['Líder', 'Municipio', 'zona_code'])
    .size()
    .reset_index(name='n_simp_zona')
)
lider_zonas = lider_zonas.merge(
    analisis_zona[['municipio_nombre', 'zona', 'votos_candidata_zona', 'votos_partido_zona', 'votos_totales_zona']],
    left_on=['Municipio', 'zona_code'],
    right_on=['municipio_nombre', 'zona'],
    how='left'
)
lider_votos = lider_zonas.groupby('Líder').agg(
    votos_candidata_en_zonas=('votos_candidata_zona', 'sum'),
    votos_partido_en_zonas=('votos_partido_zona', 'sum'),
    votos_totales_en_zonas=('votos_totales_zona', 'sum')
).reset_index()

# Merge
lider_analisis = lider_sim.merge(lider_votos, on='Líder', how='left')
lider_analisis['ratio_votos_candidata_vs_simp'] = (
    lider_analisis['votos_candidata_en_zonas'] / lider_analisis['simpatizantes'].replace(0, float('nan'))
).round(2)

print(f'=== EFECTIVIDAD POR LIDER - {CANDIDATA_NOMBRE} ===\n')
cols_lider = ['Líder', 'simpatizantes', 'municipios', 'puestos', 'votos_candidata_en_zonas',
              'votos_partido_en_zonas', 'ratio_votos_candidata_vs_simp', 'lista_municipios']
display(lider_analisis[cols_lider])

## 7. Comparativa de Candidatos en Zonas con Simpatizantes

In [ ]:
# === VOTOS POR CANDIDATO EN ZONAS DONDE HAY SIMPATIZANTES ===
# Identificar las zonas donde tenemos simpatizantes
zonas_con_simpatizantes = df_sim_valle[['Municipio', 'zona_code']].drop_duplicates()
zonas_con_simpatizantes = zonas_con_simpatizantes.rename(columns={'Municipio': 'municipio_nombre', 'zona_code': 'zona'})

# Filtrar votos de CAMARA solo en esas zonas
df_cam_en_zonas = df_camara.merge(
    zonas_con_simpatizantes,
    on=['municipio_nombre', 'zona'],
    how='inner'
)

# Ranking de candidatos preferentes en esas zonas
ranking_candidatos = (df_cam_en_zonas[df_cam_en_zonas['tipo_voto'] == 'PREFERENTE']
    .groupby(['candidato', 'partido', 'partido_nombre'])['votos']
    .sum()
    .reset_index()
    .sort_values('votos', ascending=False)
)

# Marcar candidata propia
ranking_candidatos['es_propia'] = ranking_candidatos['candidato'] == CANDIDATA_CODE

print(f'=== RANKING DE CANDIDATOS EN ZONAS CON SIMPATIZANTES ===')
print(f'(Solo votos preferentes en las {len(zonas_con_simpatizantes)} zonas donde hay simpatizantes)\n')
display(ranking_candidatos)

# Posicion de la candidata
pos = ranking_candidatos[ranking_candidatos['es_propia']].index
if len(pos) > 0:
    idx = list(ranking_candidatos.index).index(pos[0]) + 1
    votos = ranking_candidatos.loc[pos[0], 'votos']
    print(f'\n{CANDIDATA_NOMBRE} quedo en posicion #{idx} con {votos:,} votos preferentes en zonas con simpatizantes')

## 8. Visualizaciones

In [ ]:
# === GRAFICO 1: Simpatizantes vs Votos Candidata por Municipio ===

mpios_con_simp = resumen_mpio[resumen_mpio['simpatizantes'] > 0].sort_values('simpatizantes', ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(14, 8))
y_pos = range(len(mpios_con_simp))
bar_height = 0.35

bars1 = ax.barh([y - bar_height/2 for y in y_pos], mpios_con_simp['simpatizantes'], bar_height,
                label='Simpatizantes registrados', color='#3498db', alpha=0.85)
bars2 = ax.barh([y + bar_height/2 for y in y_pos], mpios_con_simp['votos_candidata'], bar_height,
                label=f'Votos {CANDIDATA_NOMBRE}', color='#e74c3c', alpha=0.85)

ax.set_yticks(list(y_pos))
ax.set_yticklabels(mpios_con_simp['municipio'].values)
ax.set_xlabel('Cantidad')
ax.set_title(f'Simpatizantes vs Votos de {CANDIDATA_NOMBRE} por Municipio\n(Top 15 municipios con mas simpatizantes)')
ax.legend(loc='lower right')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

for bar, val in zip(bars1, mpios_con_simp['simpatizantes']):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center', fontsize=9)
for bar, val in zip(bars2, mpios_con_simp['votos_candidata']):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# === GRAFICO 2: Heatmap - Tasa de conversion por municipio/zona ===

# Filtrar solo zonas con simpatizantes en los top municipios
top_mpios = resumen_mpio[resumen_mpio['simpatizantes'] >= 10]['municipio'].tolist()
zonas_heat = analisis_zona[
    (analisis_zona['simpatizantes'] > 0) &
    (analisis_zona['municipio_nombre'].isin(top_mpios))
][['municipio_nombre', 'zona', 'simpatizantes', 'votos_candidata_zona', 'ratio_candidata_vs_simpatizantes']].copy()

if len(zonas_heat) > 0:
    # Crear tabla pivote para heatmap
    pivot = zonas_heat.pivot_table(
        values='ratio_candidata_vs_simpatizantes',
        index='municipio_nombre',
        columns='zona',
        aggfunc='first'
    )

    fig, ax = plt.subplots(figsize=(16, max(6, len(pivot) * 0.6)))
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn', center=1.0,
                linewidths=0.5, ax=ax, cbar_kws={'label': 'Votos Candidata / Simpatizantes'})
    ax.set_title(f'Ratio Votos {CANDIDATA_NOMBRE} vs Simpatizantes por Zona\n(>1 = mas votos que simpatizantes, <1 = menos votos)')
    ax.set_xlabel('Zona / Comuna')
    ax.set_ylabel('Municipio')
    plt.tight_layout()
    plt.show()
else:
    print('No hay datos suficientes para el heatmap')

In [ ]:
# === GRAFICO 3: Scatter - Simpatizantes vs Votos candidata por zona ===

zonas_scatter = analisis_zona[analisis_zona['simpatizantes'] > 0].copy()

fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(
    zonas_scatter['simpatizantes'],
    zonas_scatter['votos_candidata_zona'],
    s=zonas_scatter['votos_totales_zona'] / 200,
    alpha=0.6,
    c=zonas_scatter['ratio_candidata_vs_simpatizantes'].clip(0, 5),
    cmap='RdYlGn',
    edgecolors='gray',
    linewidth=0.5
)

# Linea de referencia 1:1
max_val = max(zonas_scatter['simpatizantes'].max(), zonas_scatter['votos_candidata_zona'].max())
ax.plot([0, max_val], [0, max_val], 'k--', alpha=0.3, label='Linea 1:1')
ax.plot([0, max_val], [0, max_val * 0.5], 'r--', alpha=0.2, label='50% conversion')

# Etiquetar puntos grandes
for _, row in zonas_scatter.nlargest(8, 'simpatizantes').iterrows():
    ax.annotate(f'{row["municipio_nombre"]} Z{int(row["zona"]) if pd.notna(row["zona"]) else "?"}',
                (row['simpatizantes'], row['votos_candidata_zona']),
                fontsize=8, alpha=0.8)

plt.colorbar(scatter, label='Ratio votos/simpatizantes')
ax.set_xlabel('Simpatizantes registrados')
ax.set_ylabel(f'Votos {CANDIDATA_NOMBRE}')
ax.set_title(f'Correlacion: Simpatizantes vs Votos por Zona\n(tamano = votos totales en la zona)')
ax.legend()
plt.tight_layout()
plt.show()

# Correlacion
corr = zonas_scatter[['simpatizantes', 'votos_candidata_zona']].corr().iloc[0, 1]
print(f'Correlacion Pearson (simpatizantes vs votos candidata): {corr:.3f}')

In [ ]:
# === GRAFICO 4: Efectividad por Lider ===

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# Panel izquierdo: Simpatizantes por lider
lider_sorted = lider_analisis.sort_values('simpatizantes', ascending=True)
colors = ['#e74c3c' if x >= 1 else '#3498db' for x in lider_sorted['ratio_votos_candidata_vs_simp'].fillna(0)]

ax1.barh(lider_sorted['Líder'], lider_sorted['simpatizantes'], color='#3498db', alpha=0.8)
ax1.set_xlabel('Cantidad de simpatizantes')
ax1.set_title('Simpatizantes por Lider')
for i, (simp, lider) in enumerate(zip(lider_sorted['simpatizantes'], lider_sorted['Líder'])):
    ax1.text(simp + 5, i, f'{simp:,}', va='center', fontsize=9)

# Panel derecho: Ratio votos/simpatizantes
ax2.barh(lider_sorted['Líder'], lider_sorted['ratio_votos_candidata_vs_simp'].fillna(0),
         color=['#27ae60' if x >= 1 else '#e74c3c' for x in lider_sorted['ratio_votos_candidata_vs_simp'].fillna(0)],
         alpha=0.8)
ax2.axvline(x=1, color='black', linestyle='--', alpha=0.5, label='1:1')
ax2.set_xlabel(f'Ratio votos {CANDIDATA_NOMBRE} / simpatizantes')
ax2.set_title('Efectividad por Lider (Votos vs Simpatizantes en sus zonas)')
ax2.legend()

plt.suptitle(f'Analisis de Lideres - {CANDIDATA_NOMBRE}', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# === GRAFICO 5: Distribucion de votos por partido en zonas con simpatizantes ===

# Top partidos por votos en zonas con simpatizantes
partido_votos = (df_cam_en_zonas[df_cam_en_zonas['tipo_voto'] == 'PREFERENTE']
    .groupby('partido_nombre')['votos']
    .sum()
    .sort_values(ascending=False)
    .head(12)
)

fig, ax = plt.subplots(figsize=(12, 6))
colors_bar = ['#e74c3c' if p == PARTIDO_MAP.get(PARTIDO_CODE, '') else '#3498db' for p in partido_votos.index]
bars = ax.bar(partido_votos.index, partido_votos.values, color=colors_bar, alpha=0.85)
ax.set_ylabel('Votos preferentes')
ax.set_title(f'Votos preferentes por partido en zonas con simpatizantes\n(rojo = {PARTIDO_NOMBRE})')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.xticks(rotation=45, ha='right')

for bar, val in zip(bars, partido_votos.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100, f'{val:,}',
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# === GRAFICO 6: Top 20 puestos de votacion con mas simpatizantes ===

top_puestos = sim_por_puesto.head(20).copy()
top_puestos['label'] = top_puestos['Municipio'] + ' - ' + top_puestos['Lugar'].str[:35]

fig, ax = plt.subplots(figsize=(14, 10))
top_rev = top_puestos.iloc[::-1]

bars = ax.barh(top_rev['label'], top_rev['simpatizantes'], color='#2ecc71', alpha=0.85)
ax.set_xlabel('Cantidad de simpatizantes')
ax.set_title(f'Top 20 Puestos de Votacion con mas Simpatizantes\n(votos zona = votos de {CANDIDATA_NOMBRE} en la zona del puesto)')

# Agregar datos de votos de la zona
for i, (_, row) in enumerate(top_rev.iterrows()):
    v = int(row.get('votos_candidata_zona', 0)) if pd.notna(row.get('votos_candidata_zona', 0)) else 0
    ax.text(row['simpatizantes'] + 2, i,
            f'{int(row["simpatizantes"])} simp | {v} votos zona',
            va='center', fontsize=8)

plt.tight_layout()
plt.show()

## 9. Exportar Resultados a Excel

In [ ]:
# === EXPORTAR RESULTADOS ===
output_file = 'Resultado_Analisis_Votaciones.xlsx'

with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
    # Hoja 1: Resumen por municipio
    cols_mpio = ['municipio', 'simpatizantes', 'votos_totales', 'votos_candidata', 'votos_partido',
                 'ratio_votos_vs_simp', 'pct_candidata', 'puestos_unicos', 'lideres_unicos']
    export_mpio = resumen_mpio[[c for c in cols_mpio if c in resumen_mpio.columns]]
    export_mpio.to_excel(writer, sheet_name='Resumen Municipio', index=False)

    # Hoja 2: Detalle por zona
    cols_zona = ['municipio_nombre', 'zona', 'simpatizantes', 'votos_totales_zona',
                 'votos_candidata_zona', 'votos_partido_zona', 'ratio_candidata_vs_simpatizantes',
                 'pct_candidata_en_zona', 'puestos_unicos']
    export_zona = analisis_zona[analisis_zona['simpatizantes'] > 0][[c for c in cols_zona if c in analisis_zona.columns]]
    export_zona = export_zona.sort_values(['municipio_nombre', 'zona'])
    export_zona.to_excel(writer, sheet_name='Detalle por Zona', index=False)

    # Hoja 3: Puestos de votacion (simpatizantes)
    cols_puesto_exp = ['Municipio', 'Lugar', 'zona_code', 'simpatizantes', 'lideres',
                       'votos_totales_zona', 'votos_candidata_zona', 'lista_lideres']
    export_puestos = sim_por_puesto[[c for c in cols_puesto_exp if c in sim_por_puesto.columns]]
    export_puestos.to_excel(writer, sheet_name='Puestos Votacion', index=False)

    # Hoja 4: Efectividad por lider
    cols_lider_exp = ['Líder', 'simpatizantes', 'municipios', 'puestos',
                      'votos_candidata_en_zonas', 'votos_partido_en_zonas',
                      'ratio_votos_candidata_vs_simp', 'lista_municipios']
    export_lider = lider_analisis[[c for c in cols_lider_exp if c in lider_analisis.columns]]
    export_lider.to_excel(writer, sheet_name='Efectividad Lideres', index=False)

    # Hoja 5: Ranking candidatos en zonas con simpatizantes
    ranking_candidatos.to_excel(writer, sheet_name='Ranking Candidatos', index=False)

    # Hoja 6: Simpatizantes individuales con datos de zona
    sim_export = df_sim_valle[['Cédula', 'Líder', 'Municipio', 'Lugar', 'zona_code', 'Barrio', 'Mesa']].copy()
    sim_export = sim_export.rename(columns={'zona_code': 'Comuna'})
    sim_export.to_excel(writer, sheet_name='Simpatizantes Detalle', index=False)

    # Formatear hojas
    workbook = writer.book
    header_format = workbook.add_format({'bold': True, 'bg_color': '#4472C4', 'font_color': 'white'})
    for sheet_name in writer.sheets:
        ws = writer.sheets[sheet_name]
        ws.set_column('A:Z', 18)

print(f'Archivo exportado: {output_file}')
print(f'Hojas: Resumen Municipio, Detalle por Zona, Puestos Votacion, Efectividad Lideres, Ranking Candidatos, Simpatizantes Detalle')